# CARACTERIZACIÓN RIGUROSA DE DESCARGA RC Y LÍMITES DEL ADC

## 1. Objetivo científico
**Estudiar** la curva de descarga transitoria de un circuito RC en el dominio del tiempo.
**Validar** el modelo ideal $V(t) = V_0 e^{-t/\tau}$ frente a la limitación física impuesta por la resolución del convertidor analógico-digital (ADC) de 10 bits.

## 2. Definición del sistema experimental
**Definir** un ensamble de alta estadística sobre un punto único para aislar errores sistemáticos instrumentales:
* $R_{nom} = 10\text{ k}\Omega$
* $C_{nom} = 470\text{ }\mu\text{F}$

## 3. Repeticiones por punto
**Realizar** para el circuito configurado:
* $K = 10$ repeticiones independientes del transitorio completo, para garantizar la convergencia estadística.

## 4. Caracterización de componentes
**4.1 Medición real con multímetro:**
**Obtener** y **Registrar** $R_{meas} \pm u_{R,meas}$ y $C_{meas} \pm u_{C,meas}$.

**4.2 Valor nominal:**
**Definir** $R_{nom}$ (por código de colores) y $C_{nom}$ (por impresión de fábrica).

**4.3 Incertidumbres nominales:**
**Calcular** basándose en las tolerancias de fábrica (ej. $\pm 5\%$ para resistores y $\pm 20\%$ para capacitores electrolíticos).

## 5. Adquisición de datos
**Armar** el circuito conmutado utilizando una plataforma Arduino Uno. 

**Para cada repetición $k$, proceder con:**

**Fase de Carga:**
1. **Cargar** el capacitor conectándolo a la fuente de $5\text{V}$.
2. **Esperar** al menos $5\tau$ para alcanzar el estado de saturación ($V_0$).
![Esquema de Carga](charge.png)

**Fase de Descarga:**
3. **Ejecutar** el script de Python para iniciar la lectura serial.
4. **Conmutar** el circuito hacia la resistencia de descarga e iniciar cronómetro ($t=0$).
5. **Guardar** los datos adquiridos en formato CSV: `time_s, voltage_V`.
![Esquema de Descarga](discharge.png)

In [ ]:
#ANALISIS
import os
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def funcion_descarga(t, V0, tau):
    return V0 * np.exp(-t / tau)

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
DATA_DIR = SCRIPT_DIR
U_INSTRUMENTAL = 5.0 / 1023.0

R_NOM = 10000.0
U_R = R_NOM * 0.05
C_NOM = 0.00047
U_C = C_NOM * 0.20

TAU_TH = R_NOM * C_NOM
U_TAU_TH = np.sqrt((C_NOM * U_R)**2 + (R_NOM * U_C)**2)

files = [f"discharge_470uF_10kohms_A{i}.csv" for i in range(1, 11)]

resultados = []
t_all, v_all = None, None

for fname in files:
    path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(path):
        continue
        
    df = pd.read_csv(path)
    t = df['time_s'].values
    v = df['voltage_V'].values
    
    if fname == "discharge_470uF_10kohms_A1.csv":
        t_all = t
        v_all = v
    
    sigma_v = np.full(len(v), U_INSTRUMENTAL)
    
    try:
        popt, pcov = curve_fit(funcion_descarga, t, v, p0=[5.0, 4.7], sigma=sigma_v, absolute_sigma=True)
        perr = np.sqrt(np.diag(pcov))
        
        v_fit = funcion_descarga(t, *popt)
        chi2 = np.sum(((v - v_fit) / sigma_v) ** 2)
        ndof = len(t) - 2
        
        resultados.append({
            'Archivo': fname,
            'V0': popt[0],
            'Tau': popt[1],
            'u_Tau': perr[1],
            'Chi2_ndof': chi2 / ndof if ndof > 0 else 0
        })
    except:
        pass

df_res = pd.DataFrame(resultados)
if not df_res.empty:
    tau_prom = df_res['Tau'].mean()
    std_tau = df_res['Tau'].std()
    u_stat = std_tau / np.sqrt(len(df_res))
    u_fit = np.sqrt(np.sum(df_res['u_Tau']**2)) / len(df_res)
    u_tot = np.sqrt(u_stat**2 + u_fit**2)
    v0_prom = df_res['V0'].mean()
    v0_err = df_res['V0'].std() / np.sqrt(len(df_res))
    chi2_prom = df_res['Chi2_ndof'].mean()

if t_all is not None:
    sigma_v_all = np.full(len(v_all), U_INSTRUMENTAL)
    popt_all, _ = curve_fit(funcion_descarga, t_all, v_all, p0=[5.0, 4.7], sigma=sigma_v_all, absolute_sigma=True)
    
    plt.rcParams['text.usetex'] = False 
    fig = plt.figure(figsize=(8, 13))
    gs = gridspec.GridSpec(3, 1, height_ratios=[2, 1, 1], hspace=0.35)
    
    ax_main = fig.add_subplot(gs[0])
    ax_diff = fig.add_subplot(gs[1])
    ax_res = fig.add_subplot(gs[2])
    
    t_smooth = np.linspace(0, max(t_all) * 1.05, 500)
    V_model_smooth = funcion_descarga(t_smooth, *popt_all)
    
    ax_main.errorbar(t_all, v_all, yerr=U_INSTRUMENTAL, fmt='o', markersize=3, color='navy', ecolor='dimgray', capsize=2, elinewidth=1, markeredgecolor='navy', alpha=0.5, label='Datos Experimentales (A1)', zorder=1)
    ax_main.plot(t_smooth, V_model_smooth, color='firebrick', linestyle='-', lw=2.5, alpha=0.9, label='Modelo Ideal: V(t) = V0 e^(-t/τ)', zorder=2)
    
    ax_main.set_ylabel("Voltaje (V)", fontsize=14)
    ax_main.set_xlabel("Tiempo (s)", fontsize=14)
    ax_main.set_xlim(-1, max(t_all) * 1.05)
    ax_main.set_ylim(-0.2, 5.5)
    ax_main.tick_params(direction='in', right=True, top=True, labelsize=12)
    
    if not df_res.empty:
        texto_parametros = (
            "Parámetros Nominales:\n"
            f"R = ({R_NOM/1000:.1f} ± {U_R/1000:.1f}) kΩ\n"
            f"C = ({C_NOM*1e6:.0f} ± {U_C*1e6:.0f}) µF\n"
            f"τ_teórico = ({TAU_TH:.2f} ± {U_TAU_TH:.2f}) s\n\n"
            "Resultados del Ensamble (N=10):\n"
            f"τ_exp = ({tau_prom:.4f} ± {u_tot:.4f}) s\n"
            f"V0_exp = ({v0_prom:.4f} ± {v0_err:.4f}) V\n"
            f"χ²/ndof prom = {chi2_prom:.2f}"
        )
    else:
         texto_parametros = "Resultados del Ensamble (N=10): No data"

    props = dict(boxstyle='round,pad=0.8', facecolor='white', alpha=0.95, edgecolor='gray')
    ax_main.text(0.48, 0.95, texto_parametros, transform=ax_main.transAxes, fontsize=11,
                 verticalalignment='top', bbox=props, zorder=3)
    
    ax_main.legend(loc='lower left', fontsize=12, framealpha=0.95, edgecolor='gray')
    
    V_model_points = funcion_descarga(t_all, *popt_all)
    diff_exp_fit = (v_all - V_model_points)
    
    ax_diff.errorbar(t_all, diff_exp_fit, yerr=U_INSTRUMENTAL, fmt='o', markersize=3, color='navy', ecolor='dimgray', capsize=2, elinewidth=1, markeredgecolor='navy', alpha=0.6)
    ax_diff.axhline(0, color='black', lw=1.5, zorder=2)
    
    ax_diff.set_ylabel("Diferencia\n$V_{exp} - V_{fit}$ (V)", fontsize=12)
    ax_diff.set_xlabel("Tiempo (s)", fontsize=14)
    ax_diff.set_xlim(-1, max(t_all) * 1.05)
    ax_diff.set_ylim(-0.15, 0.15)
    ax_diff.tick_params(direction='in', right=True, top=True, labelsize=12)
    
    r_i = diff_exp_fit / U_INSTRUMENTAL
    
    ax_res.errorbar(t_all, r_i, yerr=1.0, fmt='o', markersize=3, color='navy', ecolor='dimgray', capsize=2, elinewidth=1, markeredgecolor='navy', alpha=0.6)
    ax_res.axhline(0, color='black', lw=1.5, zorder=2)
    ax_res.axhline(2.0, color='firebrick', linestyle='--', lw=1.2, alpha=0.8)
    ax_res.axhline(-2.0, color='firebrick', linestyle='--', lw=1.2, alpha=0.8)
    
    ax_res.set_ylabel("Residuos\nnormalizados $r_i$", fontsize=12)
    ax_res.set_xlabel("Tiempo (s)", fontsize=14)
    ax_res.set_xlim(-1, max(t_all) * 1.05)
    ax_res.set_ylim(-4.5, 4.5)
    ax_res.tick_params(direction='in', right=True, top=True, labelsize=12)
    
    plt.subplots_adjust(bottom=0.08)
    plt.show()


## 6. Control de calidad de datos
Para cada curva adquirida:
* **Verificar** la monotonicidad estricta de $V(t)$.
* **Eliminar** picos de ruido anómalos o transitorios previos a la conmutación.
* **Asegurar** que la ventana temporal cubra al menos $3-5\tau$ para observar la región asintótica.

## 7. Ajuste de cada curva individual
Para cada curva $k$, **Ajustar**:
$V(t) = V_0 \exp(-t/\tau)$

**Aplicar** ponderación estadística $w_i = 1/\sigma_V^2$, donde la resolución del ADC dicta $\sigma_V = 4.883\text{ mV}$.

**Obtener:**
* $\tau^{(k)}$ y $u_{\tau}^{(k)}$
* $V_0^{(k)}$ y $u_{V_0}^{(k)}$

**Guardar además:**
* $\chi^2/ndof$
* Residuos empíricos ($V_{exp} - V_{fit}$)

## 8. Estadística por punto (Ensamble)
Para consolidar las 10 repeticiones:
**Calcular** el promedio paramétrico: $\tau_{exp}$
**Calcular** la desviación estándar: $s$
**Obtener** el error estadístico (Tipo A): $u_{stat} = s/\sqrt{10}$

## 9. Cálculo de $\tau$ teórico (Dos enfoques)
**9.1 Usando valores medidos:**
**Calcular:** $\tau_{th,meas} = R_{meas} \times C_{meas}$

**9.2 Usando valores nominales:**
**Calcular:** $\tau_{th,nom} = R_{nom} \times C_{nom}$

## 10. Comparación experimento vs teoría
**Calcular** la diferencia: $\Delta\tau = \tau_{exp} - \tau_{th}$
**Evaluar** la significancia ($Z$-score): $z = |\Delta\tau| / u_{tot}$

**Interpretar:**
* $z < 2 \rightarrow$ compatible
* $2 \leq z < 3 \rightarrow$ tensión
* $z \geq 3 \rightarrow$ discrepancia significativa

## 11. Observables derivados importantes
**Obtener** el cociente de idealidad: $Q = \tau_{exp} / \tau_{th}$ (Debe tender a $1$).
**Obtener** el coeficiente de variación: $CV = s / \tau_{exp}$ (Medida de reproducibilidad).

## 12. Análisis global (Enfoque Ensamble)
**Justificar:** Se sustituye la función $\chi^2$ global de una malla variable por un ensamble de corridas estáticas. Esto permite promediar el ruido estocástico para revelar las no idealidades sistemáticas.

## 13. Identificación de no idealidades (Efecto ADC)
Para una curva representativa:
**Analizar** la diferencia absoluta: $V_{exp} - V_{fit}$
**Analizar** los residuos normalizados: $r_i = (V_{exp} - V_{fit}) / \sigma_V$

**Identificar** regiones asintóticas donde $r_i$ forme patrones estructurados repetitivos.

## 14. Resultados esperados
**Verificar** que la constante de tiempo $\tau_{exp}$ coincida con el valor nominal (escenario ideal).
**Observar** un $\chi^2/ndof > 1$ generado artificialmente por la resolución del hardware.
**Identificar** bandas diagonales en los residuos $r_i$ a tiempos tardíos, confirmando la dominante de la cuantización.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def modelo_debye(t, V1, tau1, V2, tau2):
    return V1 * np.exp(-t / tau1) + V2 * np.exp(-t / tau2)

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
DATA_DIR = SCRIPT_DIR
U_INSTRUMENTAL = 5.0 / 1023.0

files = [f"discharge_470uF_10kohms_A{i}.csv" for i in range(1, 11)]

resultados = []
t_all, v_all = None, None

p0_debye = [4.8, 4.7, 0.1, 20.0]
bounds_debye = ([4.0, 4.0, 0.0, 15.0], [5.5, 5.5, 0.5, 100.0])

for fname in files:
    path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(path):
        continue
        
    df = pd.read_csv(path)
    t = df['time_s'].values
    v = df['voltage_V'].values
    
    if fname == "discharge_470uF_10kohms_A1.csv":
        t_all = t
        v_all = v
        
    sigma_v = np.full(len(v), U_INSTRUMENTAL)
    
    try:
        popt, pcov = curve_fit(modelo_debye, t, v, p0=p0_debye, bounds=bounds_debye, sigma=sigma_v, absolute_sigma=True)
        perr = np.sqrt(np.diag(pcov))
        
        v_fit = modelo_debye(t, *popt)
        chi2 = np.sum(((v - v_fit) / sigma_v) ** 2)
        ndof = len(t) - 4
        
        resultados.append({
            'Archivo': fname,
            'V1': popt[0],
            'Tau1': popt[1],
            'u_Tau1': perr[1],
            'V2': popt[2],
            'Tau2': popt[3],
            'u_Tau2': perr[3],
            'Chi2_ndof': chi2 / ndof if ndof > 0 else 0
        })
    except:
        pass

df_res = pd.DataFrame(resultados)
if not df_res.empty:
    tau1_prom = df_res['Tau1'].mean()
    std_tau1 = df_res['Tau1'].std()
    u_stat1 = std_tau1 / np.sqrt(len(df_res))
    u_fit1 = np.sqrt(np.sum(df_res['u_Tau1']**2)) / len(df_res)
    u_tot1 = np.sqrt(u_stat1**2 + u_fit1**2)

    tau2_prom = df_res['Tau2'].mean()
    std_tau2 = df_res['Tau2'].std()
    u_stat2 = std_tau2 / np.sqrt(len(df_res))
    u_fit2 = np.sqrt(np.sum(df_res['u_Tau2']**2)) / len(df_res)
    u_tot2 = np.sqrt(u_stat2**2 + u_fit2**2)
    chi2_prom = df_res['Chi2_ndof'].mean()

if t_all is not None:
    sigma_v_all = np.full(len(v_all), U_INSTRUMENTAL)
    popt_all, _ = curve_fit(modelo_debye, t_all, v_all, p0=p0_debye, bounds=bounds_debye, sigma=sigma_v_all, absolute_sigma=True)
    
    plt.rcParams['text.usetex'] = False
    fig = plt.figure(figsize=(8, 13))
    gs = gridspec.GridSpec(3, 1, height_ratios=[2, 1, 1], hspace=0.35)
    
    ax_main = fig.add_subplot(gs[0])
    ax_diff = fig.add_subplot(gs[1])
    ax_res = fig.add_subplot(gs[2])
    
    t_smooth = np.linspace(0, max(t_all) * 1.05, 500)
    V_model_smooth = modelo_debye(t_smooth, *popt_all)
    
    ax_main.errorbar(t_all, v_all, yerr=U_INSTRUMENTAL, fmt='o', markersize=3, color='navy', ecolor='dimgray', capsize=2, elinewidth=1, markeredgecolor='navy', alpha=0.5, label='Datos Experimentales (A1)', zorder=1)
    ax_main.plot(t_smooth, V_model_smooth, color='forestgreen', linestyle='-', lw=2.5, alpha=0.9, label='Modelo Debye (2 polos)', zorder=2)
    
    ax_main.set_ylabel("Voltaje (V)", fontsize=14)
    ax_main.set_xlabel("Tiempo (s)", fontsize=14)
    ax_main.set_xlim(-1, max(t_all) * 1.05)
    ax_main.set_ylim(-0.2, 5.5)
    ax_main.tick_params(direction='in', right=True, top=True, labelsize=12)
    
    if not df_res.empty:
        texto_parametros = (
            "Resultados Debye (N=10):\n"
            f"τ1 = ({tau1_prom:.4f} ± {u_tot1:.4f}) s\n"
            f"τ2 = ({tau2_prom:.2f} ± {u_tot2:.2f}) s\n"
            f"χ²/ndof prom = {chi2_prom:.2f}\n\n"
            "Límites de Ajuste:\n"
            "τ1 ∈ [4.0, 5.5] s\n"
            "τ2 ∈ [15.0, 100.0] s"
        )
    else:
         texto_parametros = "Resultados Debye (N=10): No data"
    
    props = dict(boxstyle='round,pad=0.8', facecolor='white', alpha=0.95, edgecolor='gray')
    ax_main.text(0.48, 0.95, texto_parametros, transform=ax_main.transAxes, fontsize=11,
                 verticalalignment='top', bbox=props, zorder=3)
    
    ax_main.legend(loc='lower left', fontsize=12, framealpha=0.95, edgecolor='gray')
    
    V_model_points = modelo_debye(t_all, *popt_all)
    diff_exp_fit = (v_all - V_model_points)
    
    ax_diff.errorbar(t_all, diff_exp_fit, yerr=U_INSTRUMENTAL, fmt='o', markersize=3, color='navy', ecolor='dimgray', capsize=2, elinewidth=1, markeredgecolor='navy', alpha=0.6)
    ax_diff.axhline(0, color='black', lw=1.5, zorder=2)
    
    ax_diff.set_ylabel("Diferencia\n$V_{exp} - V_{fit}$ (V)", fontsize=12)
    ax_diff.set_xlabel("Tiempo (s)", fontsize=14)
    ax_diff.set_xlim(-1, max(t_all) * 1.05)
    ax_diff.set_ylim(-0.15, 0.15)
    ax_diff.tick_params(direction='in', right=True, top=True, labelsize=12)
    
    r_i = diff_exp_fit / U_INSTRUMENTAL
    
    ax_res.errorbar(t_all, r_i, yerr=1.0, fmt='o', markersize=3, color='navy', ecolor='dimgray', capsize=2, elinewidth=1, markeredgecolor='navy', alpha=0.6)
    ax_res.axhline(0, color='black', lw=1.5, zorder=2)
    ax_res.axhline(2.0, color='forestgreen', linestyle='--', lw=1.2, alpha=0.8)
    ax_res.axhline(-2.0, color='forestgreen', linestyle='--', lw=1.2, alpha=0.8)
    
    ax_res.set_ylabel("Residuos\nnormalizados $r_i$", fontsize=12)
    ax_res.set_xlabel("Tiempo (s)", fontsize=14)
    ax_res.set_xlim(-1, max(t_all) * 1.05)
    ax_res.set_ylim(-4.5, 4.5)
    ax_res.tick_params(direction='in', right=True, top=True, labelsize=12)
    
    plt.subplots_adjust(bottom=0.08)
    plt.show()


## 15. Entregables del análisis
**Armar** la tabla final conteniendo:
* $\tau_{exp}, u_{tot}, z, Q, CV, \chi^2/ndof$

**Generar** las gráficas recomendadas (Panel de 3 visualizaciones):
* Ajuste macroscópico $V(t)$ empírico vs predictivo.
* Diferencia absoluta $\Delta V$.
* Residuos normalizados $r_i$ evidenciando las bandas de cuantización.

## 16. Conclusión científica
El experimento permite:
**Validar** la ley de descarga $\tau = RC$ con rigor estadístico.
**Cuantificar** la precisión metrológica del ensamble.
**Demostrar** que las divergencias al final del transitorio no indican fallas del formalismo físico, sino límites de discretización del convertidor analógico-digital (ADC).